In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(f"Connected to Workspace: {ml_client.workspace_name}")

Found the config file in: /config.json


Connected to Workspace: fraud_detection_project


In [2]:
import pandas as pd

# Helper function to make loading faster
def load_azure_csv(asset_name):
    asset = ml_client.data.get(name=asset_name, version="1")
    return pd.read_csv(asset.path)

# Load Features
X_train_imb = load_azure_csv("X-train-imb")
X_train_smote = load_azure_csv("X-train-smote")
X_test = load_azure_csv("X-test-final")

# Load Labels (and flatten to 1D array)
y_train_imb = load_azure_csv("y-train-imb").values.ravel()
y_train_smote = load_azure_csv("y-train-smote").values.ravel()
y_test = load_azure_csv("y-test-final").values.ravel()

print("All cloud assets loaded successfully!")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attem

All cloud assets loaded successfully!


In [3]:
import mlflow

# Point MLflow to Azure ML Workspace
mlflow.set_tracking_uri(ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri)

# Set the Experiment Name
experiment_name = "fraud-detection-tournament"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='', creation_time=1771958099637, experiment_id='6154cf18-a565-4006-88ec-6e5e4b3c9edc', last_update_time=None, lifecycle_stage='active', name='fraud-detection-tournament', tags={}>

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, recall_score
import mlflow
import mlflow.sklearn

# 1. Define algorithms
models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Random_Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# 2. Define scenarios
scenarios = {
    "Imbalanced": (X_train_imb, y_train_imb),
    "SMOTE": (X_train_smote, y_train_smote)
}

for scenario_name, (train_x, train_y) in scenarios.items():
    for model_name, model in models.items():
        
        with mlflow.start_run(run_name=f"{model_name}-{scenario_name}"):
            print(f"Tracking {model_name} on {scenario_name}...")
            
            # Log params
            mlflow.log_param("data_strategy", scenario_name)
            mlflow.log_param("algorithm", model_name)
            mlflow.autolog()
            
            # Fit
            model.fit(train_x, train_y)
            
            # Metrics
            probs = model.predict_proba(X_test)[:, 1]
            preds = model.predict(X_test)
            mlflow.log_metric("AUPRC", average_precision_score(y_test, probs))
            mlflow.log_metric("Recall", recall_score(y_test, preds))
            
            print(f"Run {run_id} completed successfully!")

Tracking Logistic_Regression on Imbalanced...
🏃 View run Logistic_Regression-Imbalanced at: https://uksouth.api.azureml.ms/mlflow/v2.0/subscriptions/86a6ee06-cdec-4f8f-bfc5-0e831cd3e9ca/resourceGroups/uctzyac-rg/providers/Microsoft.MachineLearningServices/workspaces/fraud_detection_project/#/experiments/6154cf18-a565-4006-88ec-6e5e4b3c9edc/runs/1e13fb35-1b90-4c2b-869f-d4bc93b9f801
🧪 View experiment at: https://uksouth.api.azureml.ms/mlflow/v2.0/subscriptions/86a6ee06-cdec-4f8f-bfc5-0e831cd3e9ca/resourceGroups/uctzyac-rg/providers/Microsoft.MachineLearningServices/workspaces/fraud_detection_project/#/experiments/6154cf18-a565-4006-88ec-6e5e4b3c9edc


2026/02/25 11:59:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


MlflowException: API request to endpoint /api/2.0/mlflow/logged-models failed with error code 404 != 200. Response body: ''

In [ ]:
# 1. Get the Experiment ID by name
experiment = mlflow.get_experiment_by_name("fraud-detection-tournament")

# 2. Search for all runs in this experiment
# This returns a pandas DataFrame automatically!
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# 3. Clean up the view (Optional)
# MLflow adds many columns (start time, tags, etc.); let's pick the important ones
columns_to_show = [
    "tags.mlflow.runName", 
    "params.algorithm", 
    "params.data_strategy", 
    "metrics.AUPRC", 
    "metrics.Recall"
]

# Display the sorted results
display(runs_df[columns_to_show].sort_values(by="metrics.AUPRC", ascending=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

fig, ax = plt.subplots(figsize=(10, 7))

# Plot the random forest for both to see the SMOTE impact clearly
for scenario_name, (train_x, train_y) in scenarios.items():
    model = RandomForestClassifier().fit(train_x, train_y)
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax, name=f"Random Forest ({scenario_name})")

plt.title("Precision-Recall Curve: Imbalanced vs SMOTE")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Replace 'best_model' with the one that had the highest AUPRC in your results_df
best_model = models["Random Forest"].fit(X_train_imb, y_train_imb) 
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Fraud'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix: Best Performing Model")
plt.show()

In [ ]:
mlflow.set_experiment("fraud-detection-tuning")
mlflow.sklearn.autolog(max_tuning_runs=10, log_models=True)

from sklearn.model_selection import RandomizedSearchCV
import numpy as np

# 1. Define the Parameter Space
param_dist = {
    'n_estimators': [100, 200, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# 2. Initialize the Randomized Search
# Use 'average_precision' (AUPRC) as the scorer for imbalanced data
rf_random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,        # Number of combinations to try
    cv=3,             # 3-fold cross-validation
    scoring='average_precision', 
    n_jobs=-1, 
    verbose=1
)

# 3. Fit with MLflow tracking
# Wrap in a parent run to keep the 'Tournament' organized
with mlflow.start_run(run_name="RandomForest-SMOTE-Tuned") as parent_run:
    print("Fine-tuning in progress...")
    rf_random_search.fit(X_train_smote, y_train_smote)
    
    # Evaluate on the final test set
    best_rf = rf_random_search.best_estimator_
    y_probs = best_rf.predict_proba(X_test)[:, 1]
    final_auprc = average_precision_score(y_test, y_probs)
    
    # Log the final 'Champion' metric
    mlflow.log_metric("final_test_AUPRC", final_auprc)
    print(f"Tuning complete! Best AUPRC: {final_auprc:.4f}")

In [ ]:
# 1. Fetch the data from MLflow
# We search both experiments where we logged our results
experiment_names = ["fraud-detection-tournament", "fraud-detection-tuning"]
runs_df = mlflow.search_runs(experiment_names=experiment_names)

# 2. Extract our two competitors
# Baseline: The best RF from the first loop
baseline_mask = (runs_df['tags.mlflow.runName'].str.contains("Random Forest-SMOTE", na=False))
# Tuned: The result from our recent Parent run
tuned_mask = (runs_df['tags.mlflow.runName'] == "RF-SMOTE-Tuned-Parent")

# 3. Create a comparison table
comparison_results = []

for mask, label in [(baseline_mask, "Baseline RF (SMOTE)"), (tuned_mask, "Fine-Tuned RF (SMOTE)")]:
    run = runs_df[mask].iloc[0]
    # Check if we have AUPRC or final_test_AUPRC (depending on which tag you used)
    auprc = run.get('metrics.AUPRC') if pd.notnull(run.get('metrics.AUPRC')) else run.get('metrics.final_test_AUPRC')
    recall = run.get('metrics.Recall')
    
    comparison_results.append({
        "Model Version": label,
        "AUPRC (Precision-Recall)": auprc,
        "Recall": recall,
        "Run ID": run['run_id']
    })

final_comparison_df = pd.DataFrame(comparison_results)

# 4. Display results and determine the Winner
display(final_comparison_df)

# Automatic logic to pick the winner
best_idx = final_comparison_df['AUPRC (Precision-Recall)'].idxmax()
winner_name = final_comparison_df.loc[best_idx, 'Model Version']
lift = (final_comparison_df.loc[1, 'AUPRC (Precision-Recall)'] - final_comparison_df.loc[0, 'AUPRC (Precision-Recall)']) / final_comparison_df.loc[0, 'AUPRC (Precision-Recall)']

print("-" * 30)
print(f"THE WINNER: {winner_name}")
print(f"Performance Lift: {lift:.2%}")
print("-" * 30)

In [ ]:
import joblib
import os

# Create a folder for the model artifacts
model_path = "../models"
os.makedirs(model_path, exist_ok=True)

# Save the winning model (Random Forest + SMOTE)
joblib.dump(value=final_model, filename=f"{model_path}/fraud_rf_model.pkl")

print(f"Model saved locally at {model_path}/fraud_rf_model.pkl")

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# 1. Define the model metadata
file_model = Model(
    path=f"{model_path}/fraud_rf_model.pkl",
    type=AssetTypes.CUSTOM_MODEL,
    name="fraud-detection-rf-smote",
    description="Random Forest model trained on SMOTE-balanced credit card data.",
    version="1"
)

# 2. Register the model in the Workspace
registered_model = ml_client.models.create_or_update(file_model)

print(f"Model registered successfully!")
print(f"Name: {registered_model.name}")
print(f"Version: {registered_model.version}")